In [0]:
from pyspark.sql import functions as F

print("===== UNIT TESTS STARTED =====")

CATALOG = "databricks_project1"

BRONZE_EMPLOYEE_TABLE = f"{CATALOG}.bronze.employee_payroll"
BRONZE_LABOR_TABLE = f"{CATALOG}.bronze.labor_position"

SILVER_EMPLOYEE_TABLE = f"{CATALOG}.silver.employee_payroll"
SILVER_LABOR_TABLE = f"{CATALOG}.silver.labor_position"
SILVER_FACILITY_TABLE = f"{CATALOG}.silver.facility"

DIM_EMPLOYEE_TABLE = f"{CATALOG}.gold.dim_employee"
DIM_FACILITY_TABLE = f"{CATALOG}.gold.dim_facility"
DIM_LABOR_TABLE = f"{CATALOG}.gold.dim_labor_position"

FACT_TABLE = f"{CATALOG}.gold.fact_employee_payroll"

Load all tables

In [0]:
bronze_employee_df = spark.table(BRONZE_EMPLOYEE_TABLE)
bronze_labor_df = spark.table(BRONZE_LABOR_TABLE)

silver_employee_df = spark.table(SILVER_EMPLOYEE_TABLE)
silver_labor_df = spark.table(SILVER_LABOR_TABLE)
silver_facility_df = spark.table(SILVER_FACILITY_TABLE)

dim_employee_df = spark.table(DIM_EMPLOYEE_TABLE)
dim_facility_df = spark.table(DIM_FACILITY_TABLE)
dim_labor_df = spark.table(DIM_LABOR_TABLE)

fact_df = spark.table(FACT_TABLE)

print("All required tables loaded successfully.")

row-count tests

In [0]:
print("===== ROW COUNT TESTS =====")

expected_counts = {
    "Bronze Employee": 661,
    "Bronze Labor Position": 308,
    "Silver Employee": 651,
    "Silver Labor Position": 308,
    "Silver Facility": 57,
    "Dim Employee": 651,
    "Dim Facility": 57,
    "Dim Labor Position": 308,
    "Fact Employee Payroll": 651
}

actual_counts = {
    "Bronze Employee": bronze_employee_df.count(),
    "Bronze Labor Position": bronze_labor_df.count(),
    "Silver Employee": silver_employee_df.count(),
    "Silver Labor Position": silver_labor_df.count(),
    "Silver Facility": silver_facility_df.count(),
    "Dim Employee": dim_employee_df.count(),
    "Dim Facility": dim_facility_df.count(),
    "Dim Labor Position": dim_labor_df.count(),
    "Fact Employee Payroll": fact_df.count()
}

for table_name, expected in expected_counts.items():

    actual = actual_counts[table_name]

    status = "PASS" if actual == expected else "FAIL"

    print(
        f"{table_name}: "
        f"Expected={expected}, "
        f"Actual={actual}, "
        f"Status={status}"
    )

Employee duplicate test

In [0]:
print("===== EMPLOYEE DUPLICATE TEST =====")

duplicate_employee_count = (
    silver_employee_df
    .groupBy("Employee_Code")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate Employee_Code groups:",
    duplicate_employee_count
)

print(
    "Status:",
    "PASS" if duplicate_employee_count == 0 else "FAIL"
)

Labor Position duplicate test

In [0]:
print("===== LABOR POSITION DUPLICATE TEST =====")

duplicate_labor_count = (
    silver_labor_df
    .groupBy("Labor_Position_Code")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate Labor_Position_Code groups:",
    duplicate_labor_count
)

print(
    "Status:",
    "PASS" if duplicate_labor_count == 0 else "FAIL"
)

Facility duplicate test

In [0]:
print("===== FACILITY DUPLICATE TEST =====")

duplicate_facility_count = (
    silver_facility_df
    .groupBy("Facility_Code")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate Facility_Code groups:",
    duplicate_facility_count
)

print(
    "Status:",
    "PASS" if duplicate_facility_count == 0 else "FAIL"
)

Key NULL tests

In [0]:
print("===== KEY NULL TESTS =====")

key_null_tests = {
    "Silver Employee_Code": (
        silver_employee_df
        .filter(F.col("Employee_Code").isNull())
        .count()
    ),

    "Silver Labor_Position_Code": (
        silver_labor_df
        .filter(F.col("Labor_Position_Code").isNull())
        .count()
    ),

    "Silver Facility_Code": (
        silver_facility_df
        .filter(F.col("Facility_Code").isNull())
        .count()
    ),

    "Dim EmployeeKey": (
        dim_employee_df
        .filter(F.col("EmployeeKey").isNull())
        .count()
    ),

    "Dim Employee_Code": (
        dim_employee_df
        .filter(F.col("Employee_Code").isNull())
        .count()
    ),

    "Dim Facility_Code": (
        dim_facility_df
        .filter(F.col("Facility_Code").isNull())
        .count()
    ),

    "Dim Labor_Position_Code": (
        dim_labor_df
        .filter(F.col("Labor_Position_Code").isNull())
        .count()
    ),

    "Fact EmployeeKey": (
        fact_df
        .filter(F.col("EmployeeKey").isNull())
        .count()
    ),

    "Fact Employee_Code": (
        fact_df
        .filter(F.col("Employee_Code").isNull())
        .count()
    )
}

for test_name, null_count in key_null_tests.items():

    status = "PASS" if null_count == 0 else "FAIL"

    print(
        f"{test_name}: "
        f"NULLs={null_count}, "
        f"Status={status}"
    )

Gold primary-key uniqueness

In [0]:
print("===== GOLD PRIMARY KEY TESTS =====")

dim_employee_duplicates = (
    dim_employee_df
    .groupBy("EmployeeKey")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

dim_facility_duplicates = (
    dim_facility_df
    .groupBy("Facility_Code")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

dim_labor_duplicates = (
    dim_labor_df
    .groupBy("Labor_Position_Code")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Dim EmployeeKey duplicates:",
    dim_employee_duplicates,
    "Status:",
    "PASS" if dim_employee_duplicates == 0 else "FAIL"
)

print(
    "Dim Facility_Code duplicates:",
    dim_facility_duplicates,
    "Status:",
    "PASS" if dim_facility_duplicates == 0 else "FAIL"
)

print(
    "Dim Labor_Position_Code duplicates:",
    dim_labor_duplicates,
    "Status:",
    "PASS" if dim_labor_duplicates == 0 else "FAIL"
)

Fact Employee uniqueness

In [0]:
print("===== FACT EMPLOYEE UNIQUENESS TEST =====")

fact_employee_duplicates = (
    fact_df
    .groupBy("Employee_Code")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

fact_employee_key_duplicates = (
    fact_df
    .groupBy("EmployeeKey")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate Employee_Code in Fact:",
    fact_employee_duplicates,
    "Status:",
    "PASS" if fact_employee_duplicates == 0 else "FAIL"
)

print(
    "Duplicate EmployeeKey in Fact:",
    fact_employee_key_duplicates,
    "Status:",
    "PASS" if fact_employee_key_duplicates == 0 else "FAIL"
)

Fact → Dimension referential integrity

In [0]:
print("===== FACT → DIMENSION REFERENTIAL INTEGRITY =====")

fact_employee_missing = (
    fact_df
    .select("EmployeeKey")
    .dropDuplicates()
    .join(
        dim_employee_df
        .select("EmployeeKey")
        .dropDuplicates(),
        on="EmployeeKey",
        how="left_anti"
    )
    .count()
)

fact_facility_missing = (
    fact_df
    .select("Facility_Code")
    .dropDuplicates()
    .join(
        dim_facility_df
        .select("Facility_Code")
        .dropDuplicates(),
        on="Facility_Code",
        how="left_anti"
    )
    .count()
)

fact_labor_missing = (
    fact_df
    .select("Labor_Position_Code")
    .dropDuplicates()
    .join(
        dim_labor_df
        .select("Labor_Position_Code")
        .dropDuplicates(),
        on="Labor_Position_Code",
        how="left_anti"
    )
    .count()
)

print(
    "Fact → Dim Employee missing keys:",
    fact_employee_missing,
    "Status:",
    "PASS" if fact_employee_missing == 0 else "FAIL"
)

print(
    "Fact → Dim Facility missing keys:",
    fact_facility_missing,
    "Status:",
    "PASS" if fact_facility_missing == 0 else "FAIL"
)

print(
    "Fact → Dim Labor Position missing keys:",
    fact_labor_missing,
    "Status:",
    "PASS" if fact_labor_missing == 0 else "FAIL"
)

Verify the exact missing key

In [0]:
print("===== MISSING LABOR POSITION KEY DETAILS =====")

missing_labor_keys_df = (
    fact_df
    .select(
        "EmployeeKey",
        "Employee_Code",
        "Labor_Position_Code",
        "Labor_Position_Desc"
    )
    .dropDuplicates()
    .join(
        dim_labor_df
        .select("Labor_Position_Code")
        .dropDuplicates(),
        on="Labor_Position_Code",
        how="left_anti"
    )
)

display(missing_labor_keys_df)

Watermark validation

In [0]:
print("===== EMPLOYEE WATERMARK TEST =====")

WATERMARK_TABLE = "databricks_project1.silver.pipeline_watermark"

watermark_df = spark.table(WATERMARK_TABLE)

employee_watermark_df = (
    watermark_df
    .filter(
        (F.col("pipeline_name") == "Silver Employee") &
        (F.col("source_name") == "Employee_Payroll.xlsx")
    )
)

display(employee_watermark_df)

In [0]:
employee_watermark = (
    employee_watermark_df
    .select("last_processed_timestamp")
    .first()["last_processed_timestamp"]
)

silver_employee_max_timestamp = (
    silver_employee_df
    .agg(
        F.max("ingestion_timestamp")
        .alias("max_timestamp")
    )
    .first()["max_timestamp"]
)

print(
    "Stored Employee watermark:",
    employee_watermark
)

print(
    "Silver Employee max timestamp:",
    silver_employee_max_timestamp
)

print(
    "Watermark Status:",
    "PASS"
    if employee_watermark == silver_employee_max_timestamp
    else "FAIL"
)

Incremental-load behavior test

In [0]:
print("===== INCREMENTAL LOAD TEST =====")

employee_incremental_test_df = (
    bronze_employee_df
    .filter(
        F.col("ingestion_timestamp") >
        F.lit(employee_watermark)
    )
)

incremental_test_count = employee_incremental_test_df.count()

print(
    "Records newer than stored watermark:",
    incremental_test_count
)

print(
    "Incremental Load Status:",
    "PASS"
    if incremental_test_count == 0
    else "NEW RECORDS DETECTED"
)

In [0]:
print("===== WATERMARK DIAGNOSTIC =====")

display(
    watermark_df
    .filter(
        F.col("pipeline_name") == "Silver Employee"
    )
    .select(
        "pipeline_name",
        "source_name",
        "last_processed_timestamp",
        "updated_at"
    )
)

In [0]:
print("===== FINAL UNIT TEST SUMMARY =====")

print("Row Count Tests: PASS")
print("Duplicate Tests: PASS")
print("Key NULL Tests: PASS")
print("Dimension Uniqueness Tests: PASS")

print("Fact → Dim Employee: PASS")
print("Fact → Dim Facility: PASS")

print("Fact → Dim Labor Position: FAIL")
print("  Missing Labor_Position_Code: 5502")

print("Employee Watermark Test: PASS")
print("Incremental Load Test: PASS")

print("\n===== OVERALL UNIT TEST STATUS =====")
print("STATUS: FAILED")
print("Reason: Known Labor Position master-data issue (Code 5502)")